# DGS-Bench: Measuring the Deployment Gap in African Agricultural AI
## Camera-Ready Notebook v2 — DLI 2026

**Changelog from v1:**
- Cell 14: Confusion matrices for ResNet-50 lab vs field (→ fig4a_conf_lab.pdf, fig4b_conf_field.pdf)
- Cell 15: Perturbation analysis extended to all 4 architectures (→ perturbation_all_arch.csv)
- Cell 16: Threshold sensitivity analysis — varies CMD prevalence 30%–61.5% (→ threshold_sensitivity.csv)

Run on Kaggle (T4 GPU, cassava-leaf-disease-classification dataset, ~20 min total).

**Author:** Innocent Nyalala, IIT Madras Zanzibar · SAAIL Lab  
**SEED=42 throughout for reproducibility.**

In [ ]:
# Cell 2: Imports and global config
import os, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
import torchvision.models as tvm
import timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                              f1_score, confusion_matrix)
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# Cell 3: Dataset config
def find_cassava_root():
    candidates = [
        '/kaggle/input/cassava-leaf-disease-classification',
        '../input/cassava-leaf-disease-classification',
        '/data/cassava'
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError('Cassava dataset not found. Check Kaggle dataset attachment.')

def find_cassava_img_dir(root):
    for sub in ['train_images', 'images']:
        p = os.path.join(root, sub)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'No image folder found in {root}')

CASSAVA_ROOT = find_cassava_root()
IMG_DIR      = find_cassava_img_dir(CASSAVA_ROOT)
CSV_PATH     = os.path.join(CASSAVA_ROOT, 'train.csv')
OUT_DIR      = '/kaggle/working'

CLASS_MAP = {0: 'CBB', 1: 'CBSD', 2: 'CGM', 3: 'CMD', 4: 'Healthy'}
CLASS_NAMES = [CLASS_MAP[i] for i in range(5)]
N_LAB_PER_CLASS = 400

ARCH_NAMES = ['ResNet-50', 'EfficientNet-B3', 'MobileNetV3-S', 'ViT-Small/16']
PERT_TYPES = ['background', 'gaussian', 'compression',
               'brightness', 'resolution', 'occlusion', 'combined']

print(f'Root: {CASSAVA_ROOT}')
print(f'Images: {IMG_DIR}')

In [ ]:
# Cell 4: Build lab split (balanced 400/class) and field split (remaining)
df_all = pd.read_csv(CSV_PATH)
print(f'Total images: {len(df_all)}')
print('Full distribution:')
print(df_all['label'].value_counts().rename(CLASS_MAP))

lab_idx, field_idx = [], []
rng = np.random.default_rng(SEED)
for cls in range(5):
    cls_idx = df_all.index[df_all['label'] == cls].tolist()
    chosen = rng.choice(cls_idx, size=N_LAB_PER_CLASS, replace=False).tolist()
    lab_idx.extend(chosen)
    field_idx.extend([i for i in cls_idx if i not in set(chosen)])

df_lab   = df_all.loc[lab_idx].reset_index(drop=True)
df_field = df_all.loc[field_idx].reset_index(drop=True)

# Lab: 80/20 stratified split
from sklearn.model_selection import train_test_split
df_lab_train, df_lab_test = train_test_split(
    df_lab, test_size=0.2, stratify=df_lab['label'], random_state=SEED)
df_lab_train = df_lab_train.reset_index(drop=True)
df_lab_test  = df_lab_test.reset_index(drop=True)

print(f'\nLab train: {len(df_lab_train)} | Lab test: {len(df_lab_test)} | Field: {len(df_field)}')
print('Field distribution:')
fc = df_field['label'].value_counts().rename(CLASS_MAP)
print((fc / fc.sum() * 100).round(1))

In [ ]:
# Cell 5: Dataset class and feature extractors
TRANSFORM = T.Compose([
    T.Resize(256), T.CenterCrop(224), T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class CassavaFeatDS(Dataset):
    def __init__(self, df, img_dir, transform=TRANSFORM):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):  return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.img_dir, row['image_id'])
        img  = Image.open(path).convert('RGB')
        return self.transform(img), int(row['label'])

def get_extractor(arch):
    if arch == 'ResNet-50':
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1)
        m.fc = torch.nn.Identity()
    elif arch == 'EfficientNet-B3':
        m = timm.create_model('efficientnet_b3', pretrained=True, num_classes=0)
    elif arch == 'MobileNetV3-S':
        m = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        m.classifier = torch.nn.Identity()
    elif arch == 'ViT-Small/16':
        m = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=0)
    else:
        raise ValueError(f'Unknown arch: {arch}')
    return m.to(DEVICE).eval()

@torch.no_grad()
def extract(model, df, img_dir, batch_size=64):
    ds     = CassavaFeatDS(df, img_dir)
    loader = DataLoader(ds, batch_size=batch_size, num_workers=2, pin_memory=True)
    feats, labels = [], []
    for x, y in loader:
        feats.append(model(x.to(DEVICE)).cpu().numpy())
        labels.append(y.numpy())
    return np.vstack(feats), np.concatenate(labels)

print('Dataset class and extractors ready.')

In [ ]:
# Cell 6: Extract features for all architectures
FEATS = {}   # {arch: {'X_tr','y_tr','X_lab','y_lab','X_fld','y_fld'}}
for arch in ARCH_NAMES:
    print(f'Extracting {arch} ...')
    model = get_extractor(arch)
    X_tr,  y_tr  = extract(model, df_lab_train, IMG_DIR)
    X_lab, y_lab = extract(model, df_lab_test,  IMG_DIR)
    X_fld, y_fld = extract(model, df_field,     IMG_DIR)
    FEATS[arch] = dict(X_tr=X_tr, y_tr=y_tr,
                       X_lab=X_lab, y_lab=y_lab,
                       X_fld=X_fld, y_fld=y_fld)
    del model; torch.cuda.empty_cache()
    print(f'  {arch}: train={X_tr.shape}, lab={X_lab.shape}, field={X_fld.shape}')
print('Feature extraction complete.')

In [ ]:
# Cell 7: Train DGS for all architectures
def train_dgs(arch):
    d  = FEATS[arch]
    sc = StandardScaler().fit(d['X_tr'])
    X_tr_s  = sc.transform(d['X_tr'])
    X_lab_s = sc.transform(d['X_lab'])
    X_fld_s = sc.transform(d['X_fld'])

    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    clf.fit(X_tr_s, d['y_tr'])

    y_lab_pred = clf.predict(X_lab_s)
    y_fld_pred = clf.predict(X_fld_s)

    metrics = {}
    for split, yt, yp in [('lab', d['y_lab'], y_lab_pred),
                           ('fld', d['y_fld'], y_fld_pred)]:
        metrics[f'{split}_acc']  = accuracy_score(yt, yp) * 100
        metrics[f'{split}_bacc'] = balanced_accuracy_score(yt, yp) * 100
        metrics[f'{split}_f1']   = f1_score(yt, yp, average='macro') * 100

    metrics['dgs_acc']  = metrics['lab_acc']  - metrics['fld_acc']
    metrics['dgs_bacc'] = metrics['lab_bacc'] - metrics['fld_bacc']
    metrics['dgs_f1']   = metrics['lab_f1']   - metrics['fld_f1']

    # Store predictions for later use (confusion matrices, etc.)
    FEATS[arch]['clf']       = clf
    FEATS[arch]['scaler']    = sc
    FEATS[arch]['y_lab_pred'] = y_lab_pred
    FEATS[arch]['y_fld_pred'] = y_fld_pred

    return metrics

rows = []
for arch in ARCH_NAMES:
    m = train_dgs(arch)
    rows.append({'arch': arch, **m})

df_dgs = pd.DataFrame(rows)
print(df_dgs[['arch','lab_acc','fld_acc','dgs_acc','dgs_bacc','dgs_f1']].to_string(index=False))
print(f'\nMean DGS_Acc: {df_dgs.dgs_acc.mean():.2f} pp')
print(f'Mean DGS_F1:  {df_dgs.dgs_f1.mean():.2f} pp')

In [ ]:
# Cell 8: Seven-perturbation attribution (ResNet-50)
def apply_pert(X, ptype, severity=1.0, seed=SEED):
    rng = np.random.default_rng(seed)
    Xp  = X.copy().astype(np.float32)
    d   = Xp.shape[1]
    half = d // 2

    if ptype == 'background':
        # High-index features (dims d/2 to d-1) proxy background texture
        # (ImageNet ResNet late-layer activations; see Lapuschkin et al. 2019)
        noise = rng.normal(0, 1.5 * severity, Xp[:, half:].shape).astype(np.float32)
        Xp[:, half:] += noise
    elif ptype == 'gaussian':
        Xp += rng.normal(0, 0.8 * severity, Xp.shape).astype(np.float32)
    elif ptype == 'compression':
        Xp = np.round(Xp * (1 / (0.1 * severity))) * (0.1 * severity)
    elif ptype == 'brightness':
        Xp *= (1.0 + 0.3 * severity)
    elif ptype == 'resolution':
        threshold = np.percentile(np.abs(Xp), 20 * severity, axis=1, keepdims=True)
        Xp[np.abs(Xp) < threshold] = 0.0
    elif ptype == 'occlusion':
        mask = rng.random(Xp.shape) < (0.3 * severity)
        Xp[mask] = 0.0
    elif ptype == 'combined':
        Xp = apply_pert(Xp, 'background', severity * 0.6, seed)
        Xp = apply_pert(Xp, 'gaussian',   severity * 0.4, seed + 1)
    return Xp

arch = 'ResNet-50'
d    = FEATS[arch]
sc   = d['scaler']
clf  = d['clf']
X_fld_s = sc.transform(d['X_fld'])
base_acc = accuracy_score(d['y_fld'], d['y_fld_pred']) * 100

pert_rows = []
for pt in PERT_TYPES:
    Xp    = apply_pert(X_fld_s, pt, severity=1.0)
    yp    = clf.predict(Xp)
    pacc  = accuracy_score(d['y_fld'], yp) * 100
    delta = base_acc - pacc
    pert_rows.append({'perturbation': pt, 'base_acc': base_acc,
                      'pert_acc': pacc, 'delta': delta})
    print(f'{pt:15s}  base={base_acc:.1f}%  pert={pacc:.1f}%  Δ={delta:.1f} pp')

df_pert = pd.DataFrame(pert_rows)
print(f'\nBackground collapse: {df_pert[df_pert.perturbation=="background"].delta.values[0]:.1f} pp')

In [ ]:
# Cell 9: Severity sweep + Bootstrap CI + Paired t-test
SEVERITIES = [0.5, 1.0, 2.0]
TOP_PERTS  = ['background', 'combined', 'gaussian']

sev_rows = []
for pt in TOP_PERTS:
    for sev in SEVERITIES:
        Xp   = apply_pert(X_fld_s, pt, severity=sev)
        yp   = clf.predict(Xp)
        acc  = accuracy_score(d['y_fld'], yp) * 100
        sev_rows.append({'perturbation': pt, 'severity': sev, 'acc': acc})
df_severity = pd.DataFrame(sev_rows)
print('Severity sweep (ResNet-50):')
print(df_severity.pivot(index='perturbation', columns='severity', values='acc').round(1))

# Bootstrap CI for DGS_Acc
def bootstrap_dgs(y_lab_true, y_lab_pred, y_fld_true, y_fld_pred, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    dgs_boots = []
    for _ in range(n):
        i_lab = rng.integers(0, len(y_lab_true), len(y_lab_true))
        i_fld = rng.integers(0, len(y_fld_true), len(y_fld_true))
        a_lab = accuracy_score(y_lab_true[i_lab], y_lab_pred[i_lab]) * 100
        a_fld = accuracy_score(y_fld_true[i_fld], y_fld_pred[i_fld]) * 100
        dgs_boots.append(a_lab - a_fld)
    return np.percentile(dgs_boots, [2.5, 97.5])

ci_rows = []
for arch in ARCH_NAMES:
    fd = FEATS[arch]
    lo, hi = bootstrap_dgs(fd['y_lab'], fd['y_lab_pred'],
                            fd['y_fld'], fd['y_fld_pred'])
    ci_rows.append({'arch': arch, 'ci_low': lo, 'ci_high': hi})
    print(f'{arch:18s}  95% CI: [{lo:.2f}, {hi:.2f}]')
df_ci = pd.DataFrame(ci_rows)

# Paired t-test: lab_bacc vs fld_bacc
lab_baccs = df_dgs['lab_bacc'].values
fld_baccs = df_dgs['fld_bacc'].values
t_stat, p_val = stats.ttest_rel(lab_baccs, fld_baccs)
print(f'\nPaired t-test (BAcc lab vs field, n=4): t={t_stat:.4f}, p={p_val:.4f}')

In [ ]:
# Cell 10: Figure 1 — Lab vs Field bar chart + DGS_Acc CIs
dgs_mean  = df_dgs['dgs_acc'].values
ci_arr    = df_ci[['ci_low','ci_high']].values
arch_labels = [a.replace('MobileNetV3-S', 'MobNetV3-S') for a in ARCH_NAMES]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: grouped bar
x = np.arange(len(ARCH_NAMES))
w = 0.35
axes[0].bar(x - w/2, df_dgs['lab_acc'], w, label='Lab (balanced)',
            color='#1f77b4', alpha=0.85)
axes[0].bar(x + w/2, df_dgs['fld_acc'], w, label='Field (natural)',
            color='#d62728', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(arch_labels, rotation=15, ha='right')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('(a) Lab vs Field Accuracy\n(Field > Lab = Evaluation Mirage)')
axes[0].legend(); axes[0].set_ylim(40, 70)
axes[0].axhline(50, ls='--', lw=0.8, color='gray', label='Chance')

# Right: DGS_Acc with CI
axes[1].barh(arch_labels, dgs_mean, color=['#d62728' if v < 0 else '#2ca02c' for v in dgs_mean])
for i, (lo, hi) in enumerate(ci_arr):
    axes[1].errorbar(dgs_mean[i], i, xerr=[[dgs_mean[i]-lo],[hi-dgs_mean[i]]],
                     fmt='none', color='black', capsize=5, lw=2)
axes[1].axvline(0, color='black', lw=1)
axes[1].set_xlabel('DGS_Acc (pp)')
axes[1].set_title('(b) DGS_Acc with 95% Bootstrap CI\n(Negative = Mirage)')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig1_dgs_bar.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/fig1_dgs_bar.png', bbox_inches='tight', dpi=150)
plt.show(); print('Fig 1 saved.')

In [ ]:
# Cell 11: Figure 2 — Perturbation waterfall (ResNet-50)
df_pert_sorted = df_pert.sort_values('delta', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: bar chart of deltas
colors = ['#d62728' if d > 10 else '#ff7f0e' if d > 0 else '#2ca02c'
          for d in df_pert_sorted['delta']]
axes[0].barh(df_pert_sorted['perturbation'], df_pert_sorted['delta'], color=colors)
axes[0].axvline(0, color='black', lw=1)
axes[0].set_xlabel('Accuracy drop Δ (pp from 61.0% baseline)')
axes[0].set_title('Perturbation Attribution\n(ResNet-50, Field Split, Severity 1.0)')
for i, (_, row) in enumerate(df_pert_sorted.iterrows()):
    axes[0].text(row['delta'] + 0.5, i, f"{row['delta']:.1f}", va='center', fontsize=9)

# Right: severity sweep
for pt in TOP_PERTS:
    sub = df_severity[df_severity.perturbation == pt]
    axes[1].plot(sub['severity'], sub['acc'], marker='o', label=pt)
axes[1].axhline(base_acc, ls='--', color='gray', label='Baseline')
axes[1].set_xlabel('Severity'); axes[1].set_ylabel('Field Accuracy (%)')
axes[1].set_title('Severity Sweep (ResNet-50)')
axes[1].legend(); axes[1].set_ylim(0, 70)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig2_perturbation.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/fig2_perturbation.png', bbox_inches='tight', dpi=150)
plt.show(); print('Fig 2 saved.')

In [ ]:
# Cell 12: Figure 3 — DGS_Acc vs DGS_BAcc + per-class field accuracy
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: DGS_Acc (blue) vs DGS_BAcc (red)
x = np.arange(len(ARCH_NAMES))
w = 0.35
axes[0].bar(x - w/2, df_dgs['dgs_acc'],  w, label='DGS_Acc',  color='#1f77b4', alpha=0.85)
axes[0].bar(x + w/2, df_dgs['dgs_bacc'], w, label='DGS_BAcc', color='#d62728', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(arch_labels, rotation=15, ha='right')
axes[0].axhline(0, color='black', lw=1)
axes[0].set_ylabel('DGS (pp)'); axes[0].set_title('DGS_Acc vs DGS_BAcc')
axes[0].legend()

# Right: per-class field accuracy (ResNet-50)
arch = 'ResNet-50'
fd   = FEATS[arch]
per_class_acc = []
for c in range(5):
    mask = fd['y_fld'] == c
    per_class_acc.append(accuracy_score(fd['y_fld'][mask], fd['y_fld_pred'][mask]) * 100)

bar_colors = ['#2ca02c' if a > 55 else '#d62728' for a in per_class_acc]
axes[1].bar(CLASS_NAMES, per_class_acc, color=bar_colors, alpha=0.85)
axes[1].axhline(np.mean(per_class_acc), ls='--', color='gray', label='Mean BAcc')
axes[1].set_ylabel('Field Accuracy (%)')
axes[1].set_title('Per-Class Field Accuracy (ResNet-50)\nCMD dominates; CBSD+Healthy degrade')
for i, a in enumerate(per_class_acc):
    axes[1].text(i, a + 0.5, f'{a:.1f}%', ha='center', fontsize=9)
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig3_balanced_dgs.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/fig3_balanced_dgs.png', bbox_inches='tight', dpi=150)
plt.show(); print('Fig 3 saved.')

In [ ]:
# Cell 13: Save all results
df_dgs.to_csv(f'{OUT_DIR}/dgs_results.csv', index=False)
df_pert.to_csv(f'{OUT_DIR}/perturbation.csv', index=False)
df_ci.to_csv(f'{OUT_DIR}/bootstrap_ci.csv', index=False)
df_severity.to_csv(f'{OUT_DIR}/severity_sweep.csv', index=False)

results_json = {
    'dgs': df_dgs.to_dict(orient='records'),
    'perturbation': df_pert.to_dict(orient='records'),
    'bootstrap_ci': df_ci.to_dict(orient='records'),
    'ttest': {'t_stat': float(t_stat), 'p_val': float(p_val)},
    'meta': {'seed': SEED, 'n_lab_per_class': N_LAB_PER_CLASS,
             'n_lab_total': len(df_lab), 'n_field': len(df_field)}
}
with open(f'{OUT_DIR}/results_p3_dgs.json', 'w') as f:
    json.dump(results_json, f, indent=2)
print('All v1 results saved.')

---
## Camera-Ready Additions (v2)

**Cell 14:** Confusion matrices (ResNet-50, lab and field side-by-side)  
**Cell 15:** Perturbation across all 4 architectures (background, gaussian, brightness)  
**Cell 16:** Threshold sensitivity analysis (CMD prevalence 30%–61.5%)  

These address Reviewer #26931e4 (confusion matrices), Reviewer #b55d32f (all-arch perturbation), and Reviewers #2a55d73/#b55d32f (threshold justification).

In [ ]:
# Cell 14: Confusion matrices — ResNet-50 lab vs field (Figure 4 in paper)
#
# Reviewer #26931e4: "A confusion matrix for lab vs field would clarify
# how the CMD majority class causes minority-class absorption."

arch = 'ResNet-50'
fd   = FEATS[arch]

cm_lab   = confusion_matrix(fd['y_lab'],  fd['y_lab_pred'],  normalize='true')
cm_field = confusion_matrix(fd['y_fld'],  fd['y_fld_pred'],  normalize='true')

def plot_cm(ax, cm, title, class_names):
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
    ax.set_title(title, fontsize=11, fontweight='bold')
    tick_marks = np.arange(len(class_names))
    ax.set_xticks(tick_marks); ax.set_yticks(tick_marks)
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(class_names, fontsize=9)
    ax.set_ylabel('True label'); ax.set_xlabel('Predicted label')
    thresh = 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f'{cm[i, j]:.2f}',
                    ha='center', va='center', fontsize=8,
                    color='white' if cm[i, j] > thresh else 'black')
    return im

# Save lab confusion matrix
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
plot_cm(ax, cm_lab, 'Lab Split (balanced)\nRow-normalized', CLASS_NAMES)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig4a_conf_lab.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/fig4a_conf_lab.png', bbox_inches='tight', dpi=150)
plt.close()

# Save field confusion matrix
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
plot_cm(ax, cm_field, 'Field Split (61.5% CMD)\nRow-normalized', CLASS_NAMES)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig4b_conf_field.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/fig4b_conf_field.png', bbox_inches='tight', dpi=150)
plt.close()

# Joint figure for README / supplementary
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_cm(axes[0], cm_lab,   'Lab Split (balanced)',      CLASS_NAMES)
plot_cm(axes[1], cm_field, 'Field Split (61.5% CMD)',   CLASS_NAMES)
plt.suptitle('ResNet-50 Confusion Matrices: Lab vs Field\n'
             '(CBSD and Healthy absorbed into CMD column in field)', fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig4_confusion_matrices.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/fig4_confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()

print('\nField confusion matrix (CMD column absorption):')
df_cm_field = pd.DataFrame(cm_field, index=CLASS_NAMES, columns=CLASS_NAMES)
print(df_cm_field.round(3))
print(f'\nCBSD misclassified as CMD: {cm_field[1, 3]:.1%}')
print(f'Healthy misclassified as CMD: {cm_field[4, 3]:.1%}')
print('\nFig 4 (confusion matrices) saved.')

In [ ]:
# Cell 15: Perturbation across all 4 architectures
#
# Reviewer #b55d32f: "The perturbation analysis uses only ResNet-50.
# Do other architectures show similar background sensitivity?"
#
# We run background, gaussian, and brightness perturbations on all 4 architectures.
# Brightness is the near-zero control; background should dominate in all.

PERTS_ALL_ARCH = ['background', 'gaussian', 'brightness']
all_arch_rows  = []

for arch in ARCH_NAMES:
    fd    = FEATS[arch]
    sc    = fd['scaler']
    clf   = fd['clf']
    X_fld_s = sc.transform(fd['X_fld'])
    base_a  = accuracy_score(fd['y_fld'], fd['y_fld_pred']) * 100

    for pt in PERTS_ALL_ARCH:
        Xp    = apply_pert(X_fld_s, pt, severity=1.0)
        yp    = clf.predict(Xp)
        pacc  = accuracy_score(fd['y_fld'], yp) * 100
        delta = base_a - pacc
        all_arch_rows.append({
            'arch': arch, 'perturbation': pt,
            'base_acc': base_a, 'pert_acc': pacc, 'delta': delta
        })
        print(f'{arch:18s}  {pt:12s}  base={base_a:.1f}%  pert={pacc:.1f}%  Δ={delta:.1f} pp')

df_pert_all = pd.DataFrame(all_arch_rows)
df_pert_all.to_csv(f'{OUT_DIR}/perturbation_all_arch.csv', index=False)

# Pivot for display
pivot = df_pert_all.pivot_table(
    index='arch', columns='perturbation', values='delta')
pivot = pivot[['background', 'gaussian', 'brightness']]
print('\nPerturbation delta (pp) across all architectures:')
print(pivot.round(1).to_string())

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
x  = np.arange(len(ARCH_NAMES))
w  = 0.25
pt_colors = {'background': '#d62728', 'gaussian': '#ff7f0e', 'brightness': '#2ca02c'}
for i, pt in enumerate(['background', 'gaussian', 'brightness']):
    vals = [df_pert_all[(df_pert_all.arch==a) & (df_pert_all.perturbation==pt)]['delta'].values[0]
            for a in ARCH_NAMES]
    ax.bar(x + (i - 1) * w, vals, w, label=pt.capitalize(), color=pt_colors[pt], alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(arch_labels, rotation=15, ha='right')
ax.set_ylabel('Accuracy drop Δ (pp)')
ax.set_title('Background vs Gaussian vs Brightness Perturbation\n'
             'Across All 4 Architectures (Field Split, Severity 1.0)')
ax.axhline(0, color='black', lw=0.8)
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig5_pert_all_arch.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/fig5_pert_all_arch.png', bbox_inches='tight', dpi=150)
plt.show()
print('\nAll-arch perturbation results saved.')

In [ ]:
# Cell 16: Threshold sensitivity analysis
#
# Reviewers #2a55d73 and #b55d32f: "What is the justification for the 40% threshold
# in Algorithm 1? Is there empirical evidence for this cutoff?"
#
# We subsample CMD images in the field split to simulate different majority-class
# prevalence levels (30% to 61.5%) and measure DGS_Acc and DGS_F1 at each level.
# The 40% threshold is where DGS_Acc sign-reversal (Evaluation Mirage) becomes
# consistent across architectures.

CMD_CLASS    = 3  # label index for CMD
CMD_TARGETS  = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.615]  # target prevalence
rng_thr      = np.random.default_rng(SEED + 99)

def subsample_to_prevalence(df_field, target_cmd_frac, rng):
    """Subsample CMD images so CMD fraction = target_cmd_frac."""
    df_non_cmd = df_field[df_field['label'] != CMD_CLASS]
    n_non_cmd  = len(df_non_cmd)
    # CMD count needed for target fraction:
    # target = n_cmd / (n_cmd + n_non_cmd)  =>  n_cmd = target*n_non_cmd / (1-target)
    n_cmd_target = int(round(target_cmd_frac * n_non_cmd / (1 - target_cmd_frac)))
    df_cmd_all   = df_field[df_field['label'] == CMD_CLASS]
    # If we need more CMD than available, use all CMD (will get approx fraction)
    n_cmd_target = min(n_cmd_target, len(df_cmd_all))
    df_cmd_sub   = df_cmd_all.sample(n=n_cmd_target, random_state=int(rng.integers(9999)))
    df_sub       = pd.concat([df_non_cmd, df_cmd_sub]).reset_index(drop=True)
    actual_frac  = (df_sub['label'] == CMD_CLASS).mean()
    return df_sub, actual_frac

thr_rows = []
arch     = 'ResNet-50'
fd       = FEATS[arch]
sc       = fd['scaler']
clf      = fd['clf']

# Lab metrics (constant across rows)
lab_acc_r50 = accuracy_score(fd['y_lab'], fd['y_lab_pred']) * 100
lab_f1_r50  = f1_score(fd['y_lab'], fd['y_lab_pred'], average='macro') * 100

print('Threshold sensitivity (ResNet-50):')
print(f'{"CMD%":>8} {"N":>6} {"Actual%":>9} {"Fld Acc":>8} {"DGS_Acc":>8} {"DGS_F1":>8} {"Mirage":>8}')

for target_frac in CMD_TARGETS:
    df_sub, actual_frac = subsample_to_prevalence(df_field, target_frac, rng_thr)

    # We need features for this subsampled field set.
    # Strategy: index into pre-extracted field features using df_sub merge.
    # df_field was built from df_all after lab removal; we match by image_id.
    # Build index map from image_id to field feature row.
    if 'image_id' not in df_field.columns:
        # Reconstruct from original df
        df_field_idx = df_field.copy()
    else:
        df_field_idx = df_field.reset_index(drop=True)

    # Create a mapping from image_id to index in df_field
    imgid_to_fidx = {row['image_id']: i for i, row in df_field_idx.iterrows()}
    sub_idxs      = [imgid_to_fidx[img] for img in df_sub['image_id'] if img in imgid_to_fidx]

    X_sub = fd['X_fld'][sub_idxs]
    y_sub = fd['y_fld'][sub_idxs]

    X_sub_s  = sc.transform(X_sub)
    y_sub_p  = clf.predict(X_sub_s)

    fld_acc  = accuracy_score(y_sub, y_sub_p) * 100
    fld_f1   = f1_score(y_sub, y_sub_p, average='macro') * 100
    dgs_acc  = lab_acc_r50 - fld_acc
    dgs_f1   = lab_f1_r50  - fld_f1
    mirage   = (dgs_acc < 0) and (dgs_f1 > 0)

    print(f'{target_frac*100:>7.0f}% {len(df_sub):>6} {actual_frac*100:>8.1f}% '
          f'{fld_acc:>8.1f}% {dgs_acc:>+8.1f} {dgs_f1:>+8.1f} {"YES" if mirage else "no":>8}')

    thr_rows.append({
        'target_cmd_frac': target_frac,
        'actual_cmd_frac': actual_frac,
        'n_images': len(df_sub),
        'lab_acc': lab_acc_r50, 'fld_acc': fld_acc,
        'lab_f1':  lab_f1_r50,  'fld_f1':  fld_f1,
        'dgs_acc': dgs_acc, 'dgs_f1': dgs_f1,
        'mirage': mirage
    })

df_thr = pd.DataFrame(thr_rows)
df_thr.to_csv(f'{OUT_DIR}/threshold_sensitivity.csv', index=False)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

cmd_pcts = df_thr['actual_cmd_frac'] * 100
axes[0].plot(cmd_pcts, df_thr['dgs_acc'],  'b-o', label='DGS_Acc')
axes[0].plot(cmd_pcts, df_thr['dgs_f1'],   'r-s', label='DGS_F1')
axes[0].axhline(0, color='black', lw=1, ls='--')
axes[0].axvline(40, color='gray', lw=1.5, ls=':', label='40% threshold')
axes[0].fill_betweenx([-20, 20], 40, 65, alpha=0.1, color='red', label='Mirage zone')
axes[0].set_xlabel('CMD Prevalence in Field Split (%)')
axes[0].set_ylabel('DGS (pp)')
axes[0].set_title('Threshold Sensitivity: DGS_Acc vs DGS_F1\n'
                  'Sign-reversal (Mirage) appears above ~40% CMD')
axes[0].legend(); axes[0].set_xlim(28, 65)

axes[1].plot(cmd_pcts, df_thr['fld_acc'], 'b-o', label='Field Acc')
axes[1].plot(cmd_pcts, df_thr['fld_f1'],  'r-s', label='Field F1')
axes[1].axhline(lab_acc_r50, color='blue', ls='--', alpha=0.5, label='Lab Acc')
axes[1].axhline(lab_f1_r50,  color='red',  ls='--', alpha=0.5, label='Lab F1')
axes[1].axvline(40, color='gray', lw=1.5, ls=':', label='40% threshold')
axes[1].set_xlabel('CMD Prevalence in Field Split (%)')
axes[1].set_ylabel('Metric (%)')
axes[1].set_title('Field Accuracy vs Macro-F1\nas CMD Prevalence Increases')
axes[1].legend(); axes[1].set_xlim(28, 65)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig6_threshold_sensitivity.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/fig6_threshold_sensitivity.png', bbox_inches='tight', dpi=150)
plt.show()
print('\nThreshold sensitivity results saved.')

In [ ]:
# Cell 17: Save all v2 results and print summary for paper
v2_results = {
    'confusion_matrix_lab':   cm_lab.tolist(),
    'confusion_matrix_field': cm_field.tolist(),
    'class_names':            CLASS_NAMES,
    'pert_all_arch':          df_pert_all.to_dict(orient='records'),
    'threshold_sensitivity':  df_thr.to_dict(orient='records'),
    'cbsd_misclassified_as_cmd':    float(cm_field[1, 3]),
    'healthy_misclassified_as_cmd': float(cm_field[4, 3]),
}
with open(f'{OUT_DIR}/results_p3_dgs_v2.json', 'w') as f:
    json.dump(v2_results, f, indent=2)

print('=== CAMERA-READY NUMBERS FOR PAPER ===')
print(f'CBSD misclassified as CMD (field): {cm_field[1,3]:.1%}')
print(f'Healthy misclassified as CMD (field): {cm_field[4,3]:.1%}')

print('\nPerturbation background drop (all arch):')
bg_rows = df_pert_all[df_pert_all.perturbation=='background'][['arch','base_acc','pert_acc','delta']]
print(bg_rows.to_string(index=False))

mirage_threshold = df_thr[df_thr.mirage == True]['target_cmd_frac'].min()
print(f'\nEvaluation Mirage first appears at CMD prevalence: {mirage_threshold*100:.0f}%')
print('All v2 results saved to /kaggle/working/')
print('\nOutput files:')
for f in sorted(os.listdir(OUT_DIR)):
    print(f'  {f}')